# 05 - Meta-Controller: Selecting Worker-Agent Training Context

This notebook implements the workshop experiment. A controller chooses between two worker training policies after every stage:

- **A**: stripped action-observation context.
- **B**: retained action-observation context.

API families are split into disjoint fit, controller-validation, and final-test sets. The controller sees validation metrics only. Final results are computed only on held-out API families and stream orders. The controller is deterministic and nonparametric: it selects the candidate with the highest predefined reliability-aware validation score.

## Running this notebook standalone on Colab

This notebook is self-contained. It does **not** require `01_data_prep.ipynb` to have been run
first: section 1 rebuilds the API-Bank blocks from scratch using the identical deterministic
procedure, so the blocks match those produced by `01_data_prep.ipynb` for the same seed. If a
`preprocessed.pkl` from notebook 01 is already present it is reused instead.

**Before you start:**

1. Set the runtime to a GPU: *Runtime -> Change runtime type -> A100* (or L4/H100).
   The full run trains a Llama-3.1-8B QLoRA adapter 20 times and needs roughly
   8-14 hours on an A100. A free T4 will not finish.
2. Add your Hugging Face token as a Colab secret named `HF_TOKEN`
   (key icon in the left sidebar, "Notebook access" on). `meta-llama/Llama-3.1-8B-Instruct`
   is a gated repo, so you must also have accepted its licence on the Hub.
3. Leave `USE_DRIVE = True` so results and adapters survive a disconnect. The run
   checkpoints after every stage and resumes from where it stopped if you rerun it.

**Do a dry run first.** Set `SMOKE_TEST = True` in the config cell. That swaps in a small
ungated model and tiny budgets so the entire pipeline finishes in a few minutes on a free
T4, which verifies the plumbing before you spend hours of A100 time. Then set it back to
`False` for the real run.

## 0. Colab setup

In [ ]:
# Install dependencies. Restart the runtime if Colab asks you to.
#
# transformers is pinned. Installing it unpinned pulls 5.x, whose TrainingArguments
# was refactored and rejects arguments this notebook passes (warmup_ratio among
# them). 4.57.x is also the line the seed-42 pilot in 02_train_eval.ipynb ran under,
# so pinning keeps this experiment's training config comparable with that pilot.
# trl is not used by this notebook and is no longer installed.
!pip install -q "transformers==4.57.6" "peft>=0.17,<1.0" "accelerate>=1.0" "bitsandbytes>=0.43" huggingface_hub tqdm

import importlib, subprocess, sys
for mod in ('transformers', 'peft', 'accelerate', 'bitsandbytes'):
    try:
        print(f'{mod}: {importlib.import_module(mod).__version__}')
    except Exception as exc:
        print(f'{mod}: FAILED TO IMPORT ({exc})')

import transformers
if not transformers.__version__.startswith('4.'):
    print(
        f'\nWARNING: transformers {transformers.__version__} is loaded, but this '
        'notebook targets 4.57.x. If the install above upgraded a package that was '
        'already imported, restart the runtime (Runtime -> Restart session) and run '
        'this cell again.'
    )

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# --- Hugging Face auth -------------------------------------------------------
# Order: Colab secret -> environment variable -> interactive login.
def get_hf_token():
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            return token
    except Exception:
        pass
    return os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')

HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    print('No HF_TOKEN found. Falling back to interactive login.')
    from huggingface_hub import notebook_login
    notebook_login()
    HF_TOKEN = get_hf_token()
os.environ['HF_TOKEN'] = HF_TOKEN or ''
print('HF token present:', bool(HF_TOKEN))

# --- Where results live ------------------------------------------------------
# Colab disks are wiped when the runtime recycles. Writing the experiment
# directory to Drive is what makes the run resumable across disconnects.
USE_DRIVE = True

WORK_ROOT = Path('/content')
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_ROOT = Path('/content/drive/MyDrive/trajectory_supervision')
elif USE_DRIVE:
    print('Not running on Colab; USE_DRIVE ignored and local paths used.')
    WORK_ROOT = Path.cwd()

WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_ROOT)
print('Working directory:', Path.cwd())

In [ ]:
# --- Experiment configuration -----------------------------------------------
# SMOKE_TEST swaps in a small ungated model and tiny budgets so the whole
# pipeline can be validated on a free T4 in minutes. It produces throwaway
# numbers: never report results from a smoke run.
SMOKE_TEST = False

SEED = 42
MAX_SEQ_LEN = 1024
TOKEN_MATCHED = True

if SMOKE_TEST:
    MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M-Instruct'
    LOAD_IN_4BIT = False
    BASE_EPOCHS = 1
    EVAL_MAX_SAMPLES = 4
    MAX_FIT_ENTRIES = 16          # cap fit examples per block
    BATCH_SIZE, GRAD_ACCUM_STEPS = 2, 2
    EXPERIMENT_DIR = Path('meta_controller_smoketest')
else:
    MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'
    LOAD_IN_4BIT = True
    BASE_EPOCHS = 3
    EVAL_MAX_SAMPLES = 64
    MAX_FIT_ENTRIES = None        # no cap
    BATCH_SIZE, GRAD_ACCUM_STEPS = 4, 4
    EXPERIMENT_DIR = Path('meta_controller_seed42')

LR = 2e-4

# Delete adapters that can no longer affect the run (losing candidates, and
# superseded links in a replay chain). 20 adapters at ~160 MB each will
# otherwise fill a Drive quota. The final adapter of every run is always kept.
PRUNE_ADAPTERS = True

# Adapters live alongside the experiment dir, so on Drive when USE_DRIVE is on.
# This is what makes resume real: a stage record whose adapter was lost with the
# runtime cannot be resumed, only recomputed. Pruning keeps just the adapters the
# run still needs, so the resident footprint stays near one adapter per run
# rather than all twenty (~160 MB each for rank-32 LoRA on 8B).
ADAPTER_ROOT = Path('adapters')

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Model: {MODEL_NAME}')
print(f'Smoke test: {SMOKE_TEST}')
print(f'Experiment dir: {EXPERIMENT_DIR.resolve()}')
print(f'Adapter dir: {ADAPTER_ROOT.resolve()}')
if not PRUNE_ADAPTERS:
    print('NOTE: PRUNE_ADAPTERS is off; the full run will retain ~20 adapters (several GB).')

In [ ]:
import gc
import hashlib
import inspect
import json
import pickle
import random
import re
import shutil
import time
from collections import defaultdict

import numpy as np
import torch
import transformers
from tqdm.auto import tqdm
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer,
)
from huggingface_hub import hf_hub_download

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), (
    'This notebook requires a GPU runtime. '
    'Runtime -> Change runtime type -> GPU.'
)
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({gpu_gb:.0f} GB)')
if not SMOKE_TEST and gpu_gb < 30:
    print(
        '\nWARNING: the full 8B run needs roughly 40 GB of GPU memory for '
        'QLoRA training plus generation. This GPU is likely too small; '
        'either switch to an A100/H100 or set SMOKE_TEST = True.'
    )

In [ ]:
# Tokenizer, and the condition-specific formatting used everywhere below.
# Defined before data prep because block construction needs token counts.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

SYSTEM_PROMPT = (
    "You are a helpful assistant that can use tools. "
    "When you need to call an API, use the format: "
    "[ApiName(param1='value1', param2='value2')]. "
    "After receiving the API response, use it to formulate your answer."
)

def strip_trajectory_lines(text):
    lines = []
    for line in text.split('\n'):
        s = line.strip()
        if s.startswith('API-Request:') or s.startswith('API-Response:'):
            continue
        if 'Received API Response' in line or 'Generate API Request' in line:
            continue
        lines.append(line)
    return '\n'.join(lines).strip()

def format_entry(entry, condition):
    context = strip_trajectory_lines(entry['input']) if condition == 'A' else entry['input']
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': context},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    response = f"{entry.get('output', '')}{tokenizer.eos_token}"
    return prompt + response, len(tokenizer(prompt, add_special_tokens=False)['input_ids'])

# Smoke test of the core contrast: A must drop trajectory lines, B must keep them.
_probe = {'input': 'User: hi\nAPI-Request: [Foo(x=\'1\')]\nAPI-Response: ok\nUser: then?',
          'output': "[Bar(y='2')]"}
assert 'API-Request:' not in strip_trajectory_lines(_probe['input'])
assert 'API-Request:' in format_entry(_probe, 'B')[0]
assert 'API-Request:' not in format_entry(_probe, 'A')[0]
print('Formatting contrast verified.')

## 1. Data preparation (self-contained)

Rebuilds the four API-Bank blocks directly from the Hub. This mirrors
`01_data_prep.ipynb` step for step -- same filtering, same greedy bin-packing,
same seeded 80/20 split -- so a standalone run produces the same blocks that
notebook 01 would have produced. An existing `preprocessed.pkl` from notebook 01
takes priority, and the rebuilt blocks are cached so this cell is only slow once.

In [ ]:
MIN_ENTRIES = 10
NUM_BLOCKS = 4
PREP_DIR = Path('preprocessed_data')
PREP_DIR.mkdir(parents=True, exist_ok=True)
CONTROLLER_CACHE = PREP_DIR / f'controller_blocks_seed{SEED}.pkl'

def extract_api_name(entry):
    for field in ('output', 'input'):
        match = re.search(r'\[([A-Za-z_][A-Za-z0-9_]*)\(', entry.get(field, '') or '')
        if match:
            return match.group(1)
    return 'unknown'

def build_blocks_from_hub():
    # Step for step identical to 01_data_prep.ipynb so the blocks match.
    all_raw = []
    for fname in ('training-data/lv1-train.json',
                  'training-data/lv2-train.json',
                  'training-data/lv3-train.json'):
        path = hf_hub_download(repo_id='liminghao1630/API-Bank', filename=fname,
                               repo_type='dataset', token=HF_TOKEN or None)
        with open(path) as f:
            entries = json.load(f)
        print(f'  {fname}: {len(entries)} entries')
        all_raw.extend(entries)

    for entry in all_raw:
        entry['api_name'] = extract_api_name(entry)

    filtered = [e for e in all_raw if e['api_name'] not in ('ToolSearcher', 'unknown')]
    counts = defaultdict(int)
    for entry in filtered:
        counts[entry['api_name']] += 1
    valid_apis = sorted([a for a, c in counts.items() if c >= MIN_ENTRIES])
    valid_entries = [e for e in filtered if e['api_name'] in valid_apis]
    print(f'  {len(valid_apis)} API families, {len(valid_entries)} entries after filtering')

    # Greedy bin-packing by estimated condition-B token mass.
    api_stats = {}
    for api in tqdm(valid_apis, desc='Token counting'):
        entries = [e for e in valid_entries if e['api_name'] == api]
        sample = entries[:min(20, len(entries))]
        avg_b = np.mean([len(tokenizer.encode(format_entry(e, 'B')[0])) for e in sample])
        api_stats[api] = {'count': len(entries), 'est_total_b': avg_b * len(entries)}

    block_api_lists = [[] for _ in range(NUM_BLOCKS)]
    block_totals = [0.0] * NUM_BLOCKS
    for api in sorted(valid_apis, key=lambda a: -api_stats[a]['est_total_b']):
        idx = int(np.argmin(block_totals))
        block_api_lists[idx].append(api)
        block_totals[idx] += api_stats[api]['est_total_b']

    # Reseed immediately before the shuffles: notebook 01 consumes no global
    # randomness between its seed call and this point, so this reproduces it.
    random.seed(SEED)
    built = []
    for i in range(NUM_BLOCKS):
        block_entries = [e for e in valid_entries if e['api_name'] in block_api_lists[i]]
        random.shuffle(block_entries)
        split_idx = int(len(block_entries) * 0.8)
        built.append({
            'block_id': i + 1,
            'apis': block_api_lists[i],
            'train_entries_raw': block_entries[:split_idx],
            'eval_entries_raw': block_entries[split_idx:],
        })
        print(f"  D{i+1}: {len(block_api_lists[i])} APIs, "
              f"{split_idx} train, {len(block_entries) - split_idx} eval")
    return built

# Priority: notebook 01's output -> cached rebuild -> fresh rebuild.
legacy_pkl = PREP_DIR / 'preprocessed.pkl'
if legacy_pkl.exists():
    print(f'Using existing {legacy_pkl} from 01_data_prep.ipynb')
    with open(legacy_pkl, 'rb') as f:
        data = pickle.load(f)
    blocks = data['blocks']
    assert all('train_entries_raw' in b for b in blocks), (
        'This preprocessed.pkl predates the raw-entry export. Delete it and '
        'rerun this cell to rebuild the blocks, or rerun 01_data_prep.ipynb.'
    )
    if data['config']['model_name'] != MODEL_NAME:
        print(f"  NOTE: pkl was built for {data['config']['model_name']}, "
              f'running with {MODEL_NAME}.')
elif CONTROLLER_CACHE.exists():
    print(f'Using cached blocks at {CONTROLLER_CACHE}')
    with open(CONTROLLER_CACHE, 'rb') as f:
        blocks = pickle.load(f)['blocks']
else:
    print('Building blocks from the Hub (this takes a few minutes)...')
    blocks = build_blocks_from_hub()
    with open(CONTROLLER_CACHE, 'wb') as f:
        pickle.dump({'blocks': blocks,
                     'config': {'model_name': MODEL_NAME, 'seed': SEED,
                                'num_blocks': NUM_BLOCKS, 'min_entries': MIN_ENTRIES,
                                'system_prompt': SYSTEM_PROMPT}}, f)
    print(f'Cached to {CONTROLLER_CACHE}')

NUM_BLOCKS = len(blocks)
print(f'\nModel: {MODEL_NAME}; blocks: {NUM_BLOCKS}')

## 2. Disjoint API-Family Partitions

The split is by API family, never by individual example. A family is assigned deterministically to fit, controller validation, or final test. This prevents the controller from observing examples from a family that appears in the final test set.

In [ ]:
def family_partition(api_name, seed=SEED):
    # Stable across Python processes and independent of dictionary ordering.
    digest = hashlib.sha256(f'{seed}:{api_name}'.encode('utf-8')).hexdigest()
    bucket = int(digest[:8], 16) % 10
    if bucket < 6:
        return 'fit'
    if bucket < 8:
        return 'validation'
    return 'test'

partitions = {}
for block in blocks:
    # Fit uses only the original training split. Validation and test use
    # only the original held-out split, with API families kept disjoint.
    by_partition = {'fit': [], 'validation': [], 'test': []}
    for entry in block['train_entries_raw']:
        if family_partition(entry['api_name']) == 'fit':
            by_partition['fit'].append(entry)
    for entry in block['eval_entries_raw']:
        role = family_partition(entry['api_name'])
        if role in ('validation', 'test'):
            by_partition[role].append(entry)
    partitions[block['block_id']] = by_partition
    families = {
        role: sorted({e['api_name'] for e in entries})
        for role, entries in by_partition.items()
    }
    assert not (set(families['fit']) & set(families['validation']))
    assert not (set(families['fit']) & set(families['test']))
    assert not (set(families['validation']) & set(families['test']))
    print('D{}: '.format(block['block_id']) + ', '.join(
        f'{role}={len(families[role])} families/{len(by_partition[role])} examples'
        for role in ('fit', 'validation', 'test')
    ))

# A stage with an empty partition would silently invalidate the protocol.
for bid, by_partition in partitions.items():
    for role in ('fit', 'validation', 'test'):
        assert by_partition[role], f'D{bid} has no {role} examples; check the split.'

with open(EXPERIMENT_DIR / 'split_manifest.json', 'w') as f:
    json.dump({
        str(bid): {role: sorted({e['api_name'] for e in entries})
                  for role, entries in by_partition.items()}
        for bid, by_partition in partitions.items()
    }, f, indent=2)
print(f'\nSplit manifest written to {EXPERIMENT_DIR / "split_manifest.json"}')

In [ ]:
def stable_order(entries):
    # Same example order for A and B: selection must not be affected by a
    # shuffle that happens to differ between the candidate policies.
    keyed = []
    for entry in entries:
        key = hashlib.sha256(
            '{}:{}:{}'.format(SEED, entry['api_name'], entry['input']).encode('utf-8')
        ).hexdigest()
        keyed.append((key, entry))
    return [entry for _, entry in sorted(keyed)]

def prepare_training_entries(entries):
    # Equalize observed training tokens by capping B to A's token budget.
    # This is a transparent cost control, not a claim that the samples are
    # identical: B may use fewer examples because each example is longer.
    ordered = stable_order(entries)
    if MAX_FIT_ENTRIES is not None:
        ordered = ordered[:MAX_FIT_ENTRIES]
    formatted = {cond: [format_entry(e, cond) for e in ordered] for cond in ('A', 'B')}

    def budget(cond):
        return sum(min(len(tokenizer.encode(text, add_special_tokens=False)), MAX_SEQ_LEN)
                   for text, _ in formatted[cond])

    if not TOKEN_MATCHED:
        return {'A': (ordered, formatted['A']), 'B': (ordered, formatted['B'])}

    target = budget('A')
    selected, total = [], 0
    for entry, (text, plen) in zip(ordered, formatted['B']):
        n_tokens = min(len(tokenizer.encode(text, add_special_tokens=False)), MAX_SEQ_LEN)
        if selected and total + n_tokens > target:
            break
        selected.append((entry, text, plen))
        total += n_tokens
    return {
        'A': (ordered, formatted['A']),
        'B': ([e for e, _, _ in selected], [(t, p) for _, t, p in selected]),
    }

## 3. Worker Model and Training Utilities

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
) if LOAD_IN_4BIT else None

LORA_CONFIG = dict(
    r=32, lora_alpha=64,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)

def _load_base():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=HF_TOKEN or None, quantization_config=bnb_config,
        device_map='auto', torch_dtype=torch.bfloat16, attn_implementation='sdpa'
    )
    if LOAD_IN_4BIT:
        model = prepare_model_for_kbit_training(model)
    return model

def load_worker(adapter_path=None):
    base = _load_base()
    if adapter_path is None:
        return get_peft_model(base, LoraConfig(**LORA_CONFIG))
    # Load a fresh base, then attach only the selected adapter.
    return PeftModel.from_pretrained(base, str(adapter_path), is_trainable=True)

class TextDataset(Dataset):
    def __init__(self, formatted, tokenizer, max_length):
        self.items = []
        for text, prompt_len in formatted:
            enc = tokenizer(text, truncation=True, max_length=max_length,
                            add_special_tokens=False)
            labels = list(enc['input_ids'])
            labels[:min(prompt_len, len(labels))] = [-100] * min(prompt_len, len(labels))
            self.items.append({
                'input_ids': enc['input_ids'],
                'attention_mask': enc['attention_mask'],
                'labels': labels,
            })
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]

class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, features):
        width = max(len(f['input_ids']) for f in features)
        pad = self.tokenizer.pad_token_id
        return {
            'input_ids': torch.tensor([f['input_ids'] + [pad] * (width - len(f['input_ids'])) for f in features]),
            'attention_mask': torch.tensor([f['attention_mask'] + [0] * (width - len(f['attention_mask'])) for f in features]),
            'labels': torch.tensor([f['labels'] + [-100] * (width - len(f['labels'])) for f in features]),
        }

def build_training_arguments(**kwargs):
    """Construct TrainingArguments, dropping keys this transformers cannot take.

    The version pin should make this a no-op. It exists so a version skew fails
    loudly and specifically instead of raising a bare TypeError from inside the
    trainer, and so it is obvious when a dropped argument (warmup_ratio, say) has
    silently changed the training schedule.
    """
    supported = set(inspect.signature(TrainingArguments.__init__).parameters)
    unsupported = sorted(k for k in kwargs if k not in supported)
    if unsupported:
        print(
            f'  WARNING: transformers {transformers.__version__} does not accept '
            f'{unsupported}; dropping them. Training will NOT match the configuration '
            'reported in the paper. Install transformers==4.57.6 and restart.'
        )
        kwargs = {k: v for k, v in kwargs.items() if k in supported}
    return TrainingArguments(**kwargs)

def train_worker(model, formatted, stage_name):
    dataset = TextDataset(formatted, tokenizer, MAX_SEQ_LEN)
    args = build_training_arguments(
        output_dir=f'/tmp/meta_controller_{stage_name}',
        num_train_epochs=BASE_EPOCHS, per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS, learning_rate=LR,
        bf16=True, logging_steps=10, save_strategy='no', report_to='none',
        optim='paged_adamw_8bit' if LOAD_IN_4BIT else 'adamw_torch',
        warmup_ratio=0.1, lr_scheduler_type='cosine',
        seed=SEED, dataloader_pin_memory=True, dataloader_num_workers=2,
        gradient_checkpointing=True
    )
    result = Trainer(
        model=model, train_dataset=dataset, args=args,
        data_collator=CausalLMCollator(tokenizer),
    ).train()
    return float(result.training_loss)

## 4. Validation and Final-Test Metrics

In [ ]:
CALL_RE = re.compile(r'\[\s*([A-Za-z_][A-Za-z0-9_]*)\((.*?)\)\s*\]', re.DOTALL)
PARAM_RE = re.compile(r"(\w+)='([^']*)'")

def parse_api_call(text):
    match = CALL_RE.search(text)
    if not match:
        return None, None
    return match.group(1), {k: v for k, v in PARAM_RE.findall(match.group(2))}

def normalize_params(params):
    return {k.strip().lower(): v.strip().lower() for k, v in params.items()}

def make_subset(entries, seed):
    scored = [e for e in entries if parse_api_call(e.get('output', ''))[0] is not None]
    if len(scored) <= EVAL_MAX_SAMPLES:
        return scored
    rng = random.Random(seed)
    return [scored[i] for i in sorted(rng.sample(range(len(scored)), EVAL_MAX_SAMPLES))]

def build_generation_prompt(entry, condition):
    context = strip_trajectory_lines(entry['input']) if condition == 'A' else entry['input']
    return tokenizer.apply_chat_template([
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': context},
    ], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def evaluate_entries(model, entries, condition, sample_seed):
    was_training = model.training
    model.eval()
    entries = make_subset(entries, sample_seed)
    counts = {'total': len(entries), 'exact_full': 0, 'name': 0,
              'valid': 0, 'malformed_or_no_call': 0, 'wrong_api': 0,
              'wrong_params': 0}
    for entry in tqdm(entries, desc=f'eval[{condition}]', leave=False):
        expected_api, expected_params = parse_api_call(entry.get('output', ''))
        expected_params = normalize_params(expected_params)
        prompt = build_generation_prompt(entry, condition)
        enc = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LEN - 128,
                        return_tensors='pt').to(model.device)
        generated = model.generate(**enc, max_new_tokens=128, do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(generated[0][enc['input_ids'].shape[1]:],
                                skip_special_tokens=True)
        predicted_api, predicted_params = parse_api_call(text)
        if predicted_api is None:
            counts['malformed_or_no_call'] += 1
            continue
        counts['valid'] += 1
        if predicted_api.lower() != expected_api.lower():
            counts['wrong_api'] += 1
            continue
        counts['name'] += 1
        if normalize_params(predicted_params) == expected_params:
            counts['exact_full'] += 1
        else:
            counts['wrong_params'] += 1
    total = counts['total'] or 1
    counts.update({
        'exact_acc': counts['exact_full'] / total,
        'name_acc': counts['name'] / total,
        'valid_rate': counts['valid'] / total,
        'malformed_rate': counts['malformed_or_no_call'] / total,
        'wrong_api_rate': counts['wrong_api'] / total,
    })
    if was_training:
        model.train()
    return counts

def aggregate_metrics(metrics_by_block):
    if not metrics_by_block:
        return {'exact_acc': 0.0, 'name_acc': 0.0, 'valid_rate': 0.0,
                'malformed_rate': 0.0, 'wrong_api_rate': 0.0}
    keys = ('exact_acc', 'name_acc', 'valid_rate', 'malformed_rate', 'wrong_api_rate')
    return {key: float(np.mean([m[key] for m in metrics_by_block.values()])) for key in keys}

def controller_score(aggregate):
    # Frozen before looking at final-test results. Higher is better.
    return (0.50 * aggregate['exact_acc'] + 0.20 * aggregate['name_acc']
            + 0.20 * aggregate['valid_rate']
            - 0.05 * aggregate['malformed_rate']
            - 0.05 * aggregate['wrong_api_rate'])

## 5. Controller-Driven Sequential Training

For each stage, A and B are trained from the same selected adapter. The controller evaluates both candidates on validation families from all blocks seen so far, then carries the higher-scoring candidate into the next stage. Test families are never used in this decision.

Every stage is checkpointed to `<run>/stage_<n>.json` as soon as it completes. Rerunning
a run reloads finished stages instead of retraining them, so a Colab disconnect costs at
most the stage that was in flight.

In [ ]:
def save_adapter(model, path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(path))
    tokenizer.save_pretrained(str(path))
    return str(path)

def release_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

def prune_run_adapters(run_name, keep):
    """Delete every adapter of `run_name` except those in `keep`.

    Sweeping (rather than deleting a specific loser at selection time) is what
    makes cleanup idempotent and crash-safe: adapters that were awaiting
    deletion when a Colab session dropped are collected on the next run.

    Only ever called once the run has advanced past the kept adapters, because
    a resume replays stages from the start and a cached stage still needs its
    own adapter to be on disk.
    """
    if not PRUNE_ADAPTERS:
        return
    run_adapters = ADAPTER_ROOT / run_name
    if not run_adapters.exists():
        return
    keep = {str(Path(p).resolve()) for p in keep if p}
    for child in sorted(run_adapters.iterdir()):
        if child.is_dir() and str(child.resolve()) not in keep:
            shutil.rmtree(child, ignore_errors=True)

def train_candidate(parent_adapter, block_id, condition, run_name):
    entries = partitions[block_id]['fit']
    prepared = prepare_training_entries(entries)[condition][1]
    assert len(prepared) > 0, f'No fit examples for D{block_id} policy {condition}'
    model = load_worker(parent_adapter)
    loss = train_worker(model, prepared, f'{run_name}_D{block_id}_{condition}')
    adapter_path = save_adapter(
        model, ADAPTER_ROOT / run_name / f'candidate_D{block_id}_{condition}')
    release_model(model)
    return adapter_path, loss

def evaluate_adapter(adapter_path, block_ids, role, condition, stage):
    model = load_worker(adapter_path)
    metrics = {}
    for block_id in block_ids:
        entries = partitions[block_id][role]
        metrics[str(block_id)] = evaluate_entries(
            model, entries, condition,
            sample_seed=50_000 + stage * 100 + block_id
        )
    release_model(model)
    return metrics

def _load_stage(run_dir, stage):
    # A stage is only resumable if its record AND its adapter both survive.
    path = run_dir / f'stage_{stage}.json'
    if not path.exists():
        return None
    with open(path) as f:
        record = json.load(f)
    adapter = record.get('selected_adapter_path') or record.get('adapter_path')
    if adapter and not Path(adapter).exists():
        print(f'  stage {stage} record found but adapter is gone; recomputing.')
        return None
    return record

def _save_stage(run_dir, stage, record):
    with open(run_dir / f'stage_{stage}.json', 'w') as f:
        json.dump(record, f, indent=2)

In [ ]:
def run_controller_on_canonical_order():
    # The controller is fitted only on the canonical order. Its policy
    # schedule is frozen before any held-out-order run begins.
    run_name = 'controller_canonical'
    run_dir = EXPERIMENT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    results_path = run_dir / 'controller_results.json'
    if results_path.exists():
        print(f'[controller] complete; reusing {results_path}')
        with open(results_path) as f:
            done = json.load(f)
        prune_run_adapters(run_name, [done['history'][-1]['selected_adapter_path']])
        return done

    order = list(range(1, NUM_BLOCKS + 1))
    parent_adapter = None
    history = []
    start = time.time()

    for stage, block_id in enumerate(order, start=1):
        print(f'\n[controller] stage {stage}/{len(order)}: D{block_id}')
        cached = _load_stage(run_dir, stage)
        if cached is not None:
            # No sweep here: later cached stages still need their adapters.
            print(f'  resumed: selected {cached["selected_policy"]}')
            history.append(cached)
            parent_adapter = cached['selected_adapter_path']
            continue

        candidates = {}
        for condition in ('A', 'B'):
            path, loss = train_candidate(parent_adapter, block_id, condition, run_name)
            validation = evaluate_adapter(
                path, order[:stage], 'validation', condition, stage
            )
            aggregate = aggregate_metrics(validation)
            candidates[condition] = {
                'adapter_path': path, 'train_loss': loss,
                'validation': validation, 'validation_aggregate': aggregate,
                'score': controller_score(aggregate),
            }
            print('  {}: score={:.4f}, exact={:.1%}, valid={:.1%}, malformed={:.1%}'.format(
                condition, candidates[condition]['score'], aggregate['exact_acc'],
                aggregate['valid_rate'], aggregate['malformed_rate']))

        # Deterministic tie-break: A. This rule is frozen before testing.
        selected = max(('A', 'B'), key=lambda c: (candidates[c]['score'], c == 'A'))
        record = {
            'stage': stage, 'block_id': block_id,
            'candidates': candidates, 'selected_policy': selected,
            'selected_adapter_path': candidates[selected]['adapter_path'],
            'validation_only': True,
        }
        _save_stage(run_dir, stage, record)
        history.append(record)

        parent_adapter = candidates[selected]['adapter_path']
        # Safe now: this stage was computed fresh, so no later cached stage
        # can be waiting on anything earlier in the chain.
        prune_run_adapters(run_name, [parent_adapter])
        print(f'  controller selected {selected} for the next stage')

    prune_run_adapters(run_name, [parent_adapter])
    policy_schedule = [row['selected_policy'] for row in history]
    result = {
        'order_name': 'canonical_controller_fit', 'order': order, 'seed': SEED,
        'model_name': MODEL_NAME, 'smoke_test': SMOKE_TEST,
        'candidate_policies': ['A', 'B'],
        'controller_score': '0.50 exact + 0.20 name + 0.20 valid - 0.05 malformed - 0.05 wrong_api',
        'token_matched': TOKEN_MATCHED,
        'policy_schedule': policy_schedule, 'history': history,
        'total_time_seconds': time.time() - start,
    }
    with open(results_path, 'w') as f:
        json.dump(result, f, indent=2)
    return result

def replay_frozen_schedule(order, order_name, policy_schedule):
    # Held-out stream orders never expose validation or test metrics to the
    # controller. They only replay the frozen stage-wise schedule.
    assert len(order) == len(policy_schedule) == NUM_BLOCKS
    run_dir = EXPERIMENT_DIR / order_name
    run_dir.mkdir(parents=True, exist_ok=True)
    results_path = run_dir / 'results.json'
    if results_path.exists():
        print(f'[{order_name}] complete; reusing {results_path}')
        with open(results_path) as f:
            done = json.load(f)
        prune_run_adapters(order_name, [done['training_history'][-1]['adapter_path']])
        return done

    parent_adapter = None
    training_history = []
    start = time.time()

    for stage, (block_id, condition) in enumerate(zip(order, policy_schedule), start=1):
        print(f'\n[{order_name}] replay stage {stage}/{len(order)}: D{block_id} with {condition}')
        cached = _load_stage(run_dir, stage)
        if cached is not None:
            # No sweep here: later cached stages still need their adapters.
            print('  resumed')
            training_history.append(cached)
            parent_adapter = cached['adapter_path']
            continue
        path, loss = train_candidate(parent_adapter, block_id, condition, order_name)
        record = {
            'stage': stage, 'block_id': block_id, 'policy': condition,
            'validation_used': False, 'test_used': False,
            'train_loss': loss, 'adapter_path': path,
        }
        _save_stage(run_dir, stage, record)
        training_history.append(record)
        parent_adapter = path
        prune_run_adapters(order_name, [parent_adapter])

    # Final evaluation is the first point at which held-out test families
    # are touched. No test metric can influence a prior training decision.
    final_condition = policy_schedule[-1]
    model = load_worker(parent_adapter)
    final_test = {}
    for block_id in order:
        final_test[str(block_id)] = evaluate_entries(
            model, partitions[block_id]['test'], final_condition,
            sample_seed=90_000 + block_id
        )
    release_model(model)
    result = {
        'order_name': order_name, 'order': order, 'seed': SEED,
        'model_name': MODEL_NAME, 'smoke_test': SMOKE_TEST,
        'frozen_policy_schedule': policy_schedule,
        'final_eval_condition': final_condition,
        'training_history': training_history,
        'final_test_metrics': final_test,
        'final_metrics_role': 'held-out API families only',
        'validation_used': False, 'test_used_for_selection': False,
        'total_time_seconds': time.time() - start,
    }
    with open(results_path, 'w') as f:
        json.dump(result, f, indent=2)
    return result

## 6. Held-Out Stream-Order Evaluation

The controller is run without access to test metrics under each order. Report the reverse and rotated orders as held-out stream-order evaluations, alongside the canonical order for comparison.

This is the long cell. It performs 20 QLoRA training runs in total. Rerun it after a
disconnect and it will pick up from the last completed stage.

In [ ]:
STREAM_ORDERS = {
    'canonical': list(range(1, NUM_BLOCKS + 1)),
    'reverse_heldout': list(range(NUM_BLOCKS, 0, -1)),
    'rotate_heldout': list(range(2, NUM_BLOCKS + 1)) + [1],
}

controller_result = run_controller_on_canonical_order()
policy_schedule = controller_result['policy_schedule']
print(f'\nFrozen policy schedule: {policy_schedule}')

all_results = {
    'canonical_test_order': replay_frozen_schedule(
        STREAM_ORDERS['canonical'], 'canonical_test', policy_schedule
    ),
    'reverse_heldout': replay_frozen_schedule(
        STREAM_ORDERS['reverse_heldout'], 'reverse_heldout', policy_schedule
    ),
    'rotate_heldout': replay_frozen_schedule(
        STREAM_ORDERS['rotate_heldout'], 'rotate_heldout', policy_schedule
    ),
}

with open(EXPERIMENT_DIR / 'results_all_orders.json', 'w') as f:
    json.dump({
        'controller_fit': controller_result,
        'held_out_order_results': all_results,
    }, f, indent=2)

print('\nFinal held-out test-family summary:')
for name, result in all_results.items():
    metrics = result['final_test_metrics'].values()
    print('{}: exact={:.1%}, name={:.1%}, valid={:.1%}'.format(
          name, np.mean([m['exact_acc'] for m in metrics]),
          np.mean([m['name_acc'] for m in metrics]), np.mean([m['valid_rate'] for m in metrics])))
print(f'\nAll results written to {(EXPERIMENT_DIR / "results_all_orders.json").resolve()}')
if SMOKE_TEST:
    print('\nSMOKE TEST RUN -- these numbers are throwaway. '
          'Set SMOKE_TEST = False and use a fresh experiment dir for real results.')

## 7. Export results for download

Bundles the run's JSON artifacts into a single zip and downloads it through the
browser, so the results can be committed to the repository.

Only JSON is included. The LoRA adapters under `ADAPTER_ROOT` are deliberately left
out: they are hundreds of megabytes each, and the repository keeps bundles that size
out of git by convention. A sha256 is printed so the zip can be recorded in
`artifacts/checkpoint_manifest_seed42.md` the way the other bundles are.

This cell reads from disk rather than from variables left in memory, so it works in a
fresh runtime as long as `EXPERIMENT_DIR` points at a finished run -- for example a
Drive folder written by an earlier session.

In [ ]:
import hashlib as _hashlib
import zipfile

METRIC_KEYS = ('exact_acc', 'name_acc', 'valid_rate', 'malformed_rate', 'wrong_api_rate')

def collect_results(experiment_dir):
    experiment_dir = Path(experiment_dir)
    combined = experiment_dir / 'results_all_orders.json'
    if combined.exists():
        with open(combined) as f:
            payload = json.load(f)
        return payload.get('controller_fit'), payload.get('held_out_order_results', {}), True

    # Fall back to per-run files so a partially finished run still exports.
    controller = None
    controller_path = experiment_dir / 'controller_canonical' / 'controller_results.json'
    if controller_path.exists():
        with open(controller_path) as f:
            controller = json.load(f)
    orders = {}
    for name in ('canonical_test', 'reverse_heldout', 'rotate_heldout'):
        run_results = experiment_dir / name / 'results.json'
        if run_results.exists():
            with open(run_results) as f:
                orders[name] = json.load(f)
    return controller, orders, False

def build_summary(controller, orders, combined_present):
    summary = {
        'experiment': 'meta_controller',
        'seed': SEED,
        'model_name': MODEL_NAME,
        'smoke_test': SMOKE_TEST,
        'token_matched': TOKEN_MATCHED,
        'run_complete': bool(combined_present and controller and len(orders) == 3),
        'controller_score':
            '0.50 exact + 0.20 name + 0.20 valid - 0.05 malformed - 0.05 wrong_api',
    }
    if controller:
        summary['policy_schedule'] = controller['policy_schedule']
        summary['controller_decisions'] = [{
            'stage': h['stage'],
            'block_id': h['block_id'],
            'selected_policy': h['selected_policy'],
            'candidate_scores': {c: round(v['score'], 6)
                                 for c, v in h['candidates'].items()},
            'candidate_validation_aggregate': {
                c: {k: round(v['validation_aggregate'][k], 6) for k in METRIC_KEYS}
                for c, v in h['candidates'].items()},
        } for h in controller['history']]
    summary['held_out_orders'] = {}
    for name, result in orders.items():
        rows = list(result['final_test_metrics'].values())
        summary['held_out_orders'][name] = {
            'order': result['order'],
            'frozen_policy_schedule': result['frozen_policy_schedule'],
            'final_eval_condition': result.get('final_eval_condition'),
            'test_used_for_selection': result.get('test_used_for_selection'),
            'per_block': {b: {k: round(m[k], 6) for k in METRIC_KEYS}
                          for b, m in result['final_test_metrics'].items()},
            'mean': {k: round(float(np.mean([r[k] for r in rows])), 6) for k in METRIC_KEYS},
        }
    return summary

controller, orders, combined_present = collect_results(EXPERIMENT_DIR)
if controller is None and not orders:
    raise RuntimeError(
        f'No results found under {EXPERIMENT_DIR.resolve()}. Run section 6 first, '
        'or point EXPERIMENT_DIR at a finished run.'
    )

summary = build_summary(controller, orders, combined_present)
with open(EXPERIMENT_DIR / 'controller_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# --- readable tables, ready to paste into a run ledger ---
if summary.get('controller_decisions'):
    print('Controller decisions (validation families only)\n')
    print('| stage | block | score A | score B | selected |')
    print('|---|---|---:|---:|---|')
    for d in summary['controller_decisions']:
        print('| {} | D{} | {:.4f} | {:.4f} | {} |'.format(
            d['stage'], d['block_id'], d['candidate_scores']['A'],
            d['candidate_scores']['B'], d['selected_policy']))
    print(f"\nFrozen policy schedule: {summary['policy_schedule']}")

if summary['held_out_orders']:
    print('\nFinal held-out test-family metrics (percentages)\n')
    print('| order | exact full call | API name | valid | malformed | wrong API |')
    print('|---|---:|---:|---:|---:|---:|')
    for name, o in summary['held_out_orders'].items():
        m = o['mean']
        print('| {} | {:.1f} | {:.1f} | {:.1f} | {:.1f} | {:.1f} |'.format(
            name, 100 * m['exact_acc'], 100 * m['name_acc'], 100 * m['valid_rate'],
            100 * m['malformed_rate'], 100 * m['wrong_api_rate']))

# --- bundle every JSON artifact (never the adapters) ---
tag = 'smoketest' if SMOKE_TEST else f'seed{SEED}'
bundle = Path(f'meta_controller_{tag}_results.zip')
json_files = sorted(EXPERIMENT_DIR.rglob('*.json'))
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in json_files:
        zf.write(path, Path(EXPERIMENT_DIR.name) / path.relative_to(EXPERIMENT_DIR))

digest = _hashlib.sha256(bundle.read_bytes()).hexdigest()
print(f'\nBundled {len(json_files)} JSON files into {bundle.resolve()}')
print(f'  size:   {bundle.stat().st_size / 1e3:.1f} KB')
print(f'  sha256: {digest}')
if not summary['run_complete']:
    print('\nWARNING: this run is incomplete; the bundle holds partial results only.')
if SMOKE_TEST:
    print('\nWARNING: SMOKE_TEST results. Do not commit these as real numbers.')

print(
    '\nTo add these to the repository, unzip the download into artifacts/:\n'
    f'\n    unzip {bundle.name} -d artifacts/\n'
    f'\nwhich gives artifacts/{EXPERIMENT_DIR.name}/ containing split_manifest.json,\n'
    'controller_summary.json, the per-stage records, and results_all_orders.json.\n'
    "The repo's .gitignore excludes *.zip, so commit the unzipped files, not the zip.\n"
    'Record the sha256 above in artifacts/checkpoint_manifest_seed42.md.'
)

if IN_COLAB:
    from google.colab import files
    files.download(str(bundle))
    print('\nDownload started. If the browser blocked it, allow pop-ups and rerun this')
    print(f'cell, or grab the file from the Colab file browser at {bundle.resolve()}.')
else:
    print(f'\nNot running on Colab; the bundle is at {bundle.resolve()}.')

## Reproducibility and interpretation

The saved `split_manifest.json` identifies the disjoint API families. Each order stores controller validation records, selected policies, adapter paths, and final test-family metrics. Do not use `final_test_metrics` to tune the controller score or selection rule. The experiment remains a component-level meta-agent study: the controller selects worker training policies, but it does not claim general self-improvement.

**Before reporting anything in the paper**, confirm that `results_all_orders.json` has
`smoke_test: false`, that `split_manifest.json` shows no API family in more than one role,
and that every run records `test_used_for_selection: false`. Copy `EXPERIMENT_DIR` off the
Colab runtime; only the adapters under `ADAPTER_ROOT` are disposable.